# FReD 快速开始教程

本教程展示如何使用 FReD 处理 GROMACS REST2 采样数据，从数据验证到生成训练数据集的完整工作流。

**工作流概览**：
```
REST2数据 → 验证 → MBAR输入 → MBAR计算 → 训练数据集
```

## 1. 环境设置

In [ ]:
# 导入标准库
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

# 添加 FReD 到路径
fred_root = Path.cwd().parent
sys.path.insert(0, str(fred_root / 'scripts'))

# 导入 FReD 模块
from utils import validation, io, preprocessing, mbar, visualization, resampling

# 设置绘图样式
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ 环境设置完成")
print(f"FReD 根目录: {fred_root}")

## 2. 数据概览

In [ ]:
# 设置路径
data_dir = fred_root / 'data'
output_dir = fred_root / 'outputs'
output_dir.mkdir(exist_ok=True)

# 发现副本目录
replica_dirs = sorted([d for d in data_dir.glob('rep_*') if d.is_dir()])

print(f"数据目录: {data_dir}")
print(f"找到 {len(replica_dirs)} 个副本")

for rep_dir in replica_dirs:
    files = list(rep_dir.glob('prod.*'))
    print(f"  {rep_dir.name}: {len(files)} 个文件")

## 3. 步骤0: 数据验证

In [ ]:
print("=" * 60)
print("数据验证")
print("=" * 60)

for rep_dir in replica_dirs:
    required_files = ['prod.edr', 'prod.log', 'prod.xtc', 'prod.gro']
    files_exist = {f: (rep_dir / f).exists() for f in required_files}
    all_exist = all(files_exist.values())
    status = "✓" if all_exist else "✗"
    print(f"{rep_dir.name}: {status}")

## 4. 步骤1: 准备 MBAR 输入

In [ ]:
print("提取能量矩阵...")
edr_files = [str(rep_dir / 'prod.edr') for rep_dir in replica_dirs]
u_kn, N_k, lambda_values = preprocessing.extract_multistate_energies(edr_files)
print(f"✓ 能量矩阵: {u_kn.shape}")
print(f"  Lambda 值: {lambda_values}")

In [ ]:
print("解析副本交换记录...")
log_file = replica_dirs[0] / 'prod.log'
exchange_records = preprocessing.parse_gromacs_log(log_file)
print(f"✓ 交换轮次: {len(exchange_records['exchanges'])}")

In [ ]:
# 重建状态映射
n_replicas = len(replica_dirs)
n_cycles = N_k[0]
replica_to_state = preprocessing.build_replica_state_mapping(
    exchange_records, n_replicas=n_replicas, n_cycles=n_cycles
)
print(f"✓ 状态映射: {replica_to_state.shape}")

In [ ]:
# 保存 MBAR 输入
mbar_input_file = output_dir / 'mbar_input.npz'
io.save_npz(mbar_input_file, {
    'u_kn': u_kn, 'N_k': N_k,
    'replica_to_state': replica_to_state,
    'lambda_values': lambda_values,
    'n_cycles': n_cycles, 'n_replicas': n_replicas
})
print(f"✓ 已保存: {mbar_input_file}")

## 5. 步骤2: MBAR 计算

In [ ]:
print("运行 MBAR...")
target_state = 0
mbar_obj, weights = mbar.run_mbar(u_kn, N_k, target_state=target_state)
print(f"✓ MBAR 完成, 权重和: {weights.sum():.6f}")

In [ ]:
# 计算诊断
diagnostics = mbar.compute_diagnostics(mbar_obj)
print(f"有效样本数: {diagnostics['effective_sample_number']:.1f}")
print(f"Overlap 矩阵对角线: {np.diag(diagnostics['overlap_matrix'])}")

In [ ]:
# 可视化 Overlap 矩阵
overlap = diagnostics['overlap_matrix']
plt.figure(figsize=(8, 6))
plt.imshow(overlap, cmap='viridis')
plt.colorbar(label='Overlap')
plt.title('MBAR Overlap Matrix')
plt.xlabel('State j')
plt.ylabel('State i')
plt.show()

In [ ]:
# 保存 MBAR 结果
mbar_weights_file = output_dir / 'mbar_weights.npz'
io.save_npz(mbar_weights_file, {
    'weights': weights,
    'f_k': diagnostics['f_k'],
    'df_k': diagnostics['df_k'],
    'target_state': target_state
})
print(f"✓ 已保存: {mbar_weights_file}")

## 6. 步骤3: 构建训练数据集

In [ ]:
# 重采样
n_samples = 5000
sample_indices = resampling.resample_by_weights(
    weights, n_samples=n_samples, method='multinomial', random_seed=42
)
print(f"✓ 重采样: {n_samples} 个样本")
print(f"  唯一样本: {len(np.unique(sample_indices))}")

In [ ]:
# 转换为 replica/cycle 索引
replica_indices = sample_indices // n_cycles
cycle_indices = sample_indices % n_cycles

In [ ]:
# 提取构象
xtc_paths = [str(d / 'prod.xtc') for d in replica_dirs]
top_path = str(replica_dirs[0] / 'prod.gro')
coordinates, box_vectors = resampling.extract_configurations(
    xtc_paths, top_path, sample_indices, replica_indices, cycle_indices
)
print(f"✓ 构象: {coordinates.shape}")

In [ ]:
# 提取能量
edr_paths = [str(d / 'prod.edr') for d in replica_dirs]
energies = resampling.extract_unscaled_energies(
    edr_paths, sample_indices, replica_indices, cycle_indices, target_state=0
)
print(f"✓ 能量: {energies.shape}")
print(f"  范围: [{energies.min():.1f}, {energies.max():.1f}] kJ/mol")

In [ ]:
# 保存训练数据集
dataset_file = output_dir / 'training_dataset.npz'
original_indices = np.stack([replica_indices, cycle_indices], axis=1)

io.save_npz(dataset_file, {
    'coordinates': coordinates.astype(np.float32),
    'energies': energies.astype(np.float32),
    'box': box_vectors.astype(np.float32),
    'n_atoms': coordinates.shape[1],
    'original_indices': original_indices.astype(np.int32)
})
print(f"✓ 已保存: {dataset_file}")

## 7. 结果总结

In [ ]:
print("=" * 60)
print("工作流完成！")
print("=" * 60)
print(f"\n输出目录: {output_dir}\n")

output_files = [
    ('mbar_input.npz', '能量矩阵、状态映射'),
    ('mbar_weights.npz', 'MBAR权重、自由能'),
    ('training_dataset.npz', '坐标、能量、盒子')
]

for fname, desc in output_files:
    fpath = output_dir / fname
    if fpath.exists():
        size_mb = fpath.stat().st_size / 1024 / 1024
        print(f"✓ {fname:25s} ({size_mb:6.2f} MB)  - {desc}")
    else:
        print(f"✗ {fname:25s} (缺失)")

print("\n下一步:")
print("  1. 检查 MBAR 诊断，确认 overlap 和收敛性")
print("  2. 使用 training_dataset.npz 训练生成模型")